# Notebook 12 — Feature Selection and Engineering

**Module:** 13 — Machine Learning for Biology  
**Tier:** 1 — Master  
**Notebook:** 12 of 15  
**Time estimate:** 90 minutes

> In biological ML, you almost always have more features than samples.
> Feature selection is not optional — it is the primary tool for preventing
> overfitting and for discovering biologically meaningful signals.

---
## Step 1 — Motivation

A cancer expression dataset: 200 patients, 20,000 genes. Training a classifier
directly gives 100% training accuracy and random test accuracy. The model
memorizes noise. Feature selection first reduces 20,000 genes to 200 informative
ones, makes the model tractable, and often reveals which genes actually discriminate
the subtypes.

---
## Step 2 — Intuition

**Three categories of feature selection:**

1. **Filter methods:** Score each feature independently of the model.
   Fast. Ignores feature interactions.
   - Univariate tests: t-test, ANOVA F-test, chi-squared
   - Mutual information: $I(X_j; Y) = \sum P(x,y)\log(P(x,y)/P(x)P(y))$
   - Variance threshold: remove near-zero-variance features

2. **Wrapper methods:** Evaluate subsets by training a model. Slow but
   captures interactions.
   - Recursive Feature Elimination (RFE): train → rank by importance → remove lowest
   - Forward selection: start empty → greedily add best feature at each step
   - Backward elimination: start full → greedily remove worst feature

3. **Embedded methods:** Feature selection happens during model fitting.
   - Lasso ($L_1$ regularization): zero out uninformative feature weights
   - Tree-based importance: features used early in trees have high importance
   - Elastic Net: $L_1 + L_2$ combination

**Feature engineering:**
Creating new features from existing ones.
- Interaction terms: $X_1 \times X_2$
- Log transformation: stabilizes variance in count data (RNA-seq)
- Ratio features: e.g., tumor/normal expression ratio
- Binning: continuous → categorical (expression high/medium/low)
- Aggregation: pathway-level features (mean/sum/variance within a pathway)

---
## Step 3 — Biological Background

**Highly Variable Genes (HVGs) — the biological filter:**
Before any ML, single-cell RNA-seq standard practice selects 2,000–5,000 HVGs
— genes with high biological variability vs. technical noise. HVG selection is
a biology-informed filter that reduces dimensionality before any statistical test.
Implemented in Scanpy/Seurat; the standard metric is mean-dispersion relationship
after normalization.

**SHAP (SHapley Additive exPlanations):**
- Model-agnostic feature importance based on Shapley values from game theory
- For each sample, SHAP assigns a contribution to each feature: how much
  feature $j$ moved the prediction from the baseline for this specific sample
- Globally: mean|SHAP| per feature = feature importance
- Locally: SHAP for one sample shows why a specific patient got a specific prediction
- Critical for regulatory/clinical ML: "why did the model say this patient is high-risk?"

**Pathway-level features:**
Instead of 20,000 individual genes, compute the mean/PC1 expression per pathway
(from MSigDB KEGG/Hallmark gene sets). Reduces noise, improves interpretability,
and incorporates biological prior knowledge. Common in cancer subtype studies.

**Variance inflation factor (VIF):**
When features are highly correlated (e.g., co-expressed genes), standard feature
importance is unstable — importance distributes across correlated features.
Dimensionality reduction (PCA), clustering of correlated features, or selecting
one representative per correlation cluster addresses this.

---
## Step 4 — Mathematical Explanation

**Mutual Information:**
$$I(X_j; Y) = \sum_{x,y} P(X_j=x, Y=y) \log\frac{P(X_j=x, Y=y)}{P(X_j=x)P(Y=y)}$$

- $I = 0$: feature is independent of label (useless)
- $I > 0$: knowing $X_j$ reduces uncertainty in $Y$
- Captures non-linear dependencies (unlike correlation)
- Sklearn uses $k$-NN density estimation for continuous features

**Lasso embedded selection:**
$$\hat{\mathbf{w}} = \arg\min_{\mathbf{w}} \frac{1}{n}\|\mathbf{y} - X\mathbf{w}\|_2^2 + \lambda\|\mathbf{w}\|_1$$

The $L_1$ penalty produces exact zeros. As $\lambda$ increases, more weights are
zeroed. **Lasso regularization path:** solve for a range of $\lambda$ to see which
features survive at which sparsity level.

**Recursive Feature Elimination (RFE):**
1. Train model on all $p$ features
2. Rank features by importance (coefficient magnitude, Gini importance)
3. Remove the bottom $s$ features
4. Repeat until $k$ features remain

RFECV: wrap RFE in cross-validation to select optimal $k$.

**SHAP Shapley value (formal definition):**
$$\phi_j(f, x) = \sum_{S \subseteq \mathcal{F} \setminus \{j\}} \frac{|S|!(|\mathcal{F}|-|S|-1)!}{|\mathcal{F}|!} [f_{S \cup \{j\}}(x) - f_S(x)]$$

The Shapley value is the average marginal contribution of feature $j$ across
all possible subsets $S$ of other features. Exactly computable for trees;
approximated by sampling for neural networks.

In [ ]:
# Step 6 — Python: Filter, wrapper, embedded feature selection + SHAP
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_selection import (SelectKBest, f_classif, mutual_info_classif,
                                        RFE, RFECV, VarianceThreshold)
from sklearn.linear_model import LassoCV, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ---- Generate RNA-seq-like dataset (p >> n scenario) ----
rng = np.random.default_rng(42)
n_samples, n_genes, n_classes = 200, 1000, 4
n_informative = 40  # only 4% of genes are truly informative

X = rng.standard_normal((n_samples, n_genes))
y = np.repeat(np.arange(n_classes), n_samples // n_classes)

# Plant informative genes
for c in range(n_classes):
    mask = y == c
    X[np.ix_(mask, range(c*10, c*10+10))] += 2.5  # strong subtype markers

print(f'Dataset: {n_samples} samples, {n_genes} genes')
print(f'Informative genes: {n_informative} (genes 0–39, 4 subtypes × 10 genes)')
print(f'Noise genes: {n_genes - n_informative}')
print(f'p/n ratio: {n_genes/n_samples:.1f} (highly underdetermined)\n')

# ---- FILTER: F-test vs. Mutual Information ----
scaler = StandardScaler()
X_std = scaler.fit_transform(X)

f_scores, _ = f_classif(X_std, y)
mi_scores = mutual_info_classif(X_std, y, random_state=42)

top_f = np.argsort(f_scores)[-50:][::-1]
top_mi = np.argsort(mi_scores)[-50:][::-1]

# How many of the true informative genes (0-39) are in top-50?
true_set = set(range(n_informative))
recall_f = len(set(top_f) & true_set) / n_informative
recall_mi = len(set(top_mi) & true_set) / n_informative
print(f'Filter F-test  top-50 recall of true genes: {recall_f:.0%}')
print(f'Filter MI      top-50 recall of true genes: {recall_mi:.0%}')

# ---- EMBEDDED: Lasso regularization path ----
from sklearn.linear_model import LassoCV
X_reg = X_std[:, :100]  # subset for speed (genes 0-99 include all informative)
y_binary = (y >= 2).astype(float)  # binary variant for Lasso regression demo

lasso = LassoCV(cv=5, max_iter=5000, random_state=42)
lasso.fit(X_reg, y_binary)
nz = np.sum(lasso.coef_ != 0)
true_informative_in_subset = set(range(40))
nz_selected = set(np.where(lasso.coef_ != 0)[0])
recall_lasso = len(nz_selected & true_informative_in_subset) / len(true_informative_in_subset)
print(f'\nLasso selected {nz} features (of 100). Recall true informative: {recall_lasso:.0%}')

# ---- EMBEDDED: Tree importance ----
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_std[:, :100], y)
top_rf = np.argsort(rf.feature_importances_)[-50:][::-1]
recall_rf = len(set(top_rf) & true_informative_in_subset) / len(true_informative_in_subset)
print(f'RF importance top-50 (from 100 genes) recall: {recall_rf:.0%}')

# ---- Compare: pipeline accuracy (with vs. without feature selection) ----
results = {}
for name, pipe in [
    ('All features\n(no selection)', Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, C=0.1))
    ])),
    ('F-test top-50', Pipeline([
        ('select', SelectKBest(f_classif, k=50)),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, C=1))
    ])),
    ('MI top-50', Pipeline([
        ('select', SelectKBest(mutual_info_classif, k=50)),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, C=1))
    ])),
]:
    cv_acc = cross_val_score(pipe, X, y, cv=StratifiedKFold(5), scoring='accuracy')
    results[name] = cv_acc
    print(f'{name.replace(chr(10), " ")}: CV acc={cv_acc.mean():.3f}±{cv_acc.std():.3f}')

# ---- SHAP values with RandomForest ----
try:
    import shap
    X_small = X_std[:50, :50]  # small subset for speed
    y_small = y[:50]
    rf_small = RandomForestClassifier(n_estimators=50, random_state=42)
    rf_small.fit(X_small, y_small)
    explainer = shap.TreeExplainer(rf_small)
    shap_values = explainer.shap_values(X_small)  # list of arrays, one per class
    shap_available = True
    print('\nSHAP computed successfully')
except ImportError:
    shap_available = False
    print('\nSHAP not installed — skipping SHAP panel')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

# Panel A: F-score and MI distributions
ax = axes[0]
bins = np.linspace(0, max(f_scores.max(), 1), 50)
ax.hist(f_scores, bins=50, alpha=0.6, color='steelblue', label='F-score')
ax.axvline(np.sort(f_scores)[-50], color='steelblue', ls='--', lw=1.5,
             label=f'Top-50 cutoff')
# Overlay positions of true informative genes
true_f = f_scores[:n_informative]
ax.scatter(true_f, np.full(n_informative, 5), color='tomato', s=20, zorder=5,
             label='Informative genes')
ax.set_xlabel('F-score')
ax.set_ylabel('Count')
ax.set_title(f'A. F-test distribution\n(recall of true genes: {recall_f:.0%})')
ax.legend(fontsize=7)

# Panel B: Feature importance (RF) — top 20 genes, color by whether informative
ax = axes[1]
top20_idx = np.argsort(rf.feature_importances_)[-20:][::-1]
imp_vals = rf.feature_importances_[top20_idx]
colors_bar = ['tomato' if i < n_informative else 'steelblue' for i in top20_idx]
ax.barh(range(20), imp_vals, color=colors_bar, edgecolor='black', alpha=0.8)
ax.set_yticks(range(20))
ax.set_yticklabels([f'Gene {i}' for i in top20_idx], fontsize=8)
ax.set_xlabel('Importance')
ax.set_title(f'B. RF importance (top 20)\nred=informative, blue=noise')

# Panel C: CV accuracy comparison
ax = axes[2]
names = list(results.keys())
means = [results[n].mean() for n in names]
stds = [results[n].std() for n in names]
ax.bar(range(len(names)), means, yerr=stds, capsize=4,
         color=['grey', 'steelblue', 'tomato'], edgecolor='black', alpha=0.8)
ax.set_xticks(range(len(names)))
ax.set_xticklabels([n.replace('\n', '\n') for n in names], fontsize=8)
ax.axhline(1/n_classes, color='grey', ls='--', lw=0.8, label='Random')
ax.set_ylabel('5-fold CV accuracy')
ax.set_title('C. CV accuracy\n(feature selection lifts all features baseline)')
ax.set_ylim(0, 1)
ax.legend(fontsize=7)

# Panel D: Lasso coefficient path (top features)
ax = axes[3]
from sklearn.linear_model import lasso_path
alphas, coefs, _ = lasso_path(X_std[:, :50], y_binary, n_alphas=50, max_iter=5000)
# Color coding: red if informative (gene < 40), blue if noise
for gene_idx in range(50):
    color = 'tomato' if gene_idx < n_informative else 'steelblue'
    alpha_val = 0.8 if gene_idx < n_informative else 0.1
    ax.plot(np.log10(alphas + 1e-8), coefs[gene_idx], color=color, alpha=alpha_val, lw=0.8)
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('log10(alpha)')
ax.set_ylabel('Coefficient')
ax.set_title('D. Lasso regularization path\nred=informative, blue=noise genes')
# Custom legend
from matplotlib.lines import Line2D
ax.legend([Line2D([0],[0],color='tomato'), Line2D([0],[0],color='steelblue')],
            ['Informative', 'Noise'], fontsize=8)

plt.suptitle('Module 13 NB12: Feature Selection and Engineering', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_selection.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement mutual information from scratch for discrete classes and a continuous
   feature using histogram binning. Compare to `sklearn.feature_selection.mutual_info_classif`.
2. Implement forward feature selection from scratch: start with an empty set,
   at each step add the feature with the best cross-validated accuracy.
3. Apply RFE with cross-validation (`RFECV`) to select the optimal number of
   features. Does it recover the correct 40 informative genes?
4. Create a pathway-level feature: group the 40 informative genes into 4 pathways
   of 10 genes each. Use the mean per pathway as features. Compare classification
   accuracy to individual gene features.

---
## Step 10 — Quiz

1. What is the difference between filter, wrapper, and embedded feature selection?
2. Write the Lasso objective function. Why does $L_1$ regularization produce sparsity?
3. What does mutual information measure? How does it differ from correlation?
4. What are Shapley values and why are they fair attribution measures?
5. Why must feature selection happen inside cross-validation, not before it?

---
## Step 12 — Reflection

> *[In one paragraph: explain why correlated features confound feature importance
> estimates in tree-based models, and describe how you would handle a dataset
> with 500 genes that are organized into 30 known co-expression modules.]*

---
*Next: `13_ml_for_genomics_and_sequence_data.ipynb`*